## 01 - Data Foundation

### 1. Load data & sanity checks

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = Path("..")  # vì notebook nằm trong notebooks/
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"

train_path = DATA_RAW / "train.csv"
sample_path = DATA_RAW / "sample_submission.csv"

train = pd.read_csv(train_path)
sample = pd.read_csv(sample_path)

print("train shape:", train.shape)
print("sample shape:", sample.shape)

display(train.head())
display(sample.head())
print(train.dtypes)

train shape: (711980, 8)
sample shape: (31944, 29)


/var/folders/bg/hsvxykxj7yx0jz1nrhc3k6z40000gn/T/ipykernel_7145/3633023479.py:16: DtypeWarning: Columns (0: Stt) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(train_path)


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,2000004,SKU-08063,12,242700,2184300,"123559,1",1482709
1,2020-11-17,2000003,SKU-09458,600,"131818,1818",79090909,110000,66000000
2,2020-11-18,2000007,SKU-08062,6,230000,940909,101000,606000
3,2020-11-18,2000006,SKU-09458,240,270000,44181818,110000,26400000
4,2020-11-18,2000005,SKU-09458,240,270000,44181818,110000,26400000


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,SKU-00002_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,SKU-00003_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,SKU-00004_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,SKU-00005_validation,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Date              str
Stt            object
ItemCode          str
Quantity        int64
UnitPrice         str
SalesAmount     int64
Unit Cost         str
Cost Amount       str
dtype: object


In [2]:
print("Columns in train:")
print(train.columns.tolist())

print("\nDate range:")
print(train["Date"].min(), "→", train["Date"].max())

print("\nNumber of unique SKUs in train:")
print(train["ItemCode"].nunique())

print("\nNumber of sample rows:")
print(len(sample))

print("\nSample id examples:")
print(sample["id"].head(10).tolist())

print("\nMissing values:")
display(train.isna().sum())

print("\nQuantity summary:")
display(train["Quantity"].describe())

print("\nDuplicate rows:")
print(train.duplicated().sum())

Columns in train:
['Date', 'Stt', 'ItemCode', 'Quantity', 'UnitPrice', 'SalesAmount', 'Unit Cost', 'Cost Amount']

Date range:
2020-11-17 → 2025-09-05

Number of unique SKUs in train:
15972

Number of sample rows:
31944

Sample id examples:
['SKU-00001_validation', 'SKU-00002_validation', 'SKU-00003_validation', 'SKU-00004_validation', 'SKU-00005_validation', 'SKU-00006_validation', 'SKU-00007_validation', 'SKU-00008_validation', 'SKU-00009_validation', 'SKU-00010_validation']

Missing values:


Date           0
Stt            0
ItemCode       0
Quantity       0
UnitPrice      0
SalesAmount    0
Unit Cost      0
Cost Amount    0
dtype: int64


Quantity summary:


count    711980.000000
mean          3.437250
std          25.490722
min        -998.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        5998.000000
Name: Quantity, dtype: float64


Duplicate rows:
1473


#### 1.1 Parse numeric columns

In [3]:
def parse_number_series(s):
    """
    Parse numeric columns that may contain comma decimal separators or string formatting.
    Example: '12,5' -> 12.5
    """
    if pd.api.types.is_numeric_dtype(s):
        return s
    
    return (
        s.astype(str)
         .str.strip()
         .str.replace(",", ".", regex=False)
         .str.replace(" ", "", regex=False)
         .replace({"nan": np.nan, "None": np.nan})
         .astype(float)
    )

train_clean = train.copy()

train_clean["Date"] = pd.to_datetime(train_clean["Date"])

# integer/normal numeric columns
for col in ["Quantity", "SalesAmount"]:
    train_clean[col] = pd.to_numeric(train_clean[col], errors="coerce")

# columns that may have comma decimal separators
for col in ["UnitPrice", "Unit Cost", "Cost Amount"]:
    train_clean[col] = parse_number_series(train_clean[col])

print(train_clean.dtypes)

print("\nMissing after parse:")
print(train_clean[["Quantity", "UnitPrice", "SalesAmount", "Unit Cost", "Cost Amount"]].isna().sum())

display(train_clean.head())

Date           datetime64[us]
Stt                    object
ItemCode                  str
Quantity                int64
UnitPrice             float64
SalesAmount             int64
Unit Cost             float64
Cost Amount           float64
dtype: object

Missing after parse:
Quantity       0
UnitPrice      0
SalesAmount    0
Unit Cost      0
Cost Amount    0
dtype: int64


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
0,2020-11-17,2000004,SKU-08063,12,242700.0000,2184300,123559.1,1482709.0
1,2020-11-17,2000003,SKU-09458,600,131818.1818,79090909,110000.0,66000000.0
2,2020-11-18,2000007,SKU-08062,6,230000.0000,940909,101000.0,606000.0
3,2020-11-18,2000006,SKU-09458,240,270000.0000,44181818,110000.0,26400000.0
4,2020-11-18,2000005,SKU-09458,240,270000.0000,44181818,110000.0,26400000.0


In [4]:
print("Date range:", train_clean["Date"].min(), "→", train_clean["Date"].max())
print("Number of SKUs:", train_clean["ItemCode"].nunique())

display(train_clean[["Quantity", "UnitPrice", "SalesAmount", "Unit Cost", "Cost Amount"]].describe())

Date range: 2020-11-17 00:00:00 → 2025-09-05 00:00:00
Number of SKUs: 15972


,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
count,711980.000000,7.119800e+05,7.119800e+05,7.119800e+05,7.119800e+05
mean,3.437250,5.361727e+05,9.709366e+05,3.779208e+05,7.335175e+05
std,25.490722,1.151269e+06,3.888067e+06,6.655551e+05,3.282586e+06
min,-998.000000,-3.001100e+07,-1.842400e+08,-1.979902e+07,-1.668652e+08
25%,1.000000,1.450000e+05,2.500000e+05,8.846680e+04,1.621750e+05
50%,1.000000,2.850000e+05,5.100000e+05,2.022776e+05,3.661390e+05
75%,2.000000,7.280000e+05,1.146600e+06,5.249945e+05,8.492450e+05
max,5998.000000,7.962754e+07,9.324131e+08,4.050000e+07,8.387178e+08


## 1.2 Check return transactions

In [5]:
return_mask = (
    (train_clean["Quantity"] < 0) &
    (train_clean["SalesAmount"] < 0) &
    (train_clean["Cost Amount"] < 0)
)

print("Return rows:", return_mask.sum())
print("Return row ratio:", return_mask.mean())

print("Positive quantity total:", train_clean.loc[train_clean["Quantity"] > 0, "Quantity"].sum())
print("Return abs quantity total:", -train_clean.loc[train_clean["Quantity"] < 0, "Quantity"].sum())

display(train_clean.loc[return_mask].head())

Return rows: 37279
Return row ratio: 0.052359616843169754
Positive quantity total: 2534091
Return abs quantity total: 86838


,Date,Stt,ItemCode,Quantity,UnitPrice,SalesAmount,Unit Cost,Cost Amount
47,2020-12-09,2000110,SKU-08063,-12,-209102.2500,-2509227,0.00,-1482709.0
48,2020-12-09,2000110,SKU-08061,-12,-231818.1667,-2781818,0.00,-1897349.0
1185,2021-05-21,2001755,SKU-07770,-18,-231818.1667,-4172727,-174000.00,-3132000.0
1317,2021-06-08,2002157,SKU-02768,-24,-161000.0000,-3864000,-85000.00,-2040000.0
1321,2021-06-08,2002162,SKU-10532,-2,-676618.0000,-1353236,-390065.25,-780131.0


## 2. Convert transaction data to daily SKU panel

### 2.1 Create target daily

In [6]:
df = train_clean.copy()

df["qty_positive"] = df["Quantity"].clip(lower=0)
df["qty_return"] = (-df["Quantity"].clip(upper=0))

df["profit"] = df["SalesAmount"] - df["Cost Amount"]

daily = (
    df.groupby(["ItemCode", "Date"], as_index=False)
      .agg(
          y_net=("Quantity", "sum"),
          y_gross=("qty_positive", "sum"),
          y_return=("qty_return", "sum"),
          sales=("SalesAmount", "sum"),
          cost=("Cost Amount", "sum"),
          profit=("profit", "sum"),
          transaction_count=("Stt", "count")
      )
)

daily["y_net_clip"] = daily["y_net"].clip(lower=0)

print("daily shape:", daily.shape)
display(daily.head())
display(daily[["y_net", "y_gross", "y_return", "sales", "cost", "profit"]].describe())

daily shape: (507050, 10)


,ItemCode,Date,y_net,y_gross,y_return,sales,cost,profit,transaction_count,y_net_clip
0,SKU-00001,2025-05-26,1,1,0,830368,0.0,830368.0,1,1
1,SKU-00001,2025-05-27,1,1,0,1032840,0.0,1032840.0,1,1
2,SKU-00001,2025-05-28,2,2,0,2084484,0.0,2084484.0,2,2
3,SKU-00001,2025-05-30,2,2,0,1414600,0.0,1414600.0,2,2
4,SKU-00001,2025-06-04,6,6,0,4138320,0.0,4138320.0,6,6


,y_net,y_gross,y_return,sales,cost,profit
count,507050.000000,507050.000000,507050.000000,5.070500e+05,5.070500e+05,5.070500e+05
mean,4.826453,4.997714,0.171261,1.363352e+06,1.029977e+06,3.333747e+05
std,40.754966,40.801293,3.544906,5.249288e+06,4.386714e+06,1.708470e+06
min,-998.000000,0.000000,0.000000,-3.720000e+07,-5.082637e+07,-1.291655e+08
25%,1.000000,1.000000,0.000000,2.891000e+05,1.980000e+05,6.964825e+04
50%,2.000000,2.000000,0.000000,6.501000e+05,4.771980e+05,1.519055e+05
75%,3.000000,3.000000,0.000000,1.425000e+06,1.098942e+06,3.305120e+05
max,10358.000000,10358.000000,998.000000,9.324131e+08,8.387178e+08,2.007427e+08


### 2.2 Create full calendar for all SKU

In [7]:
all_skus = pd.Index(sorted(train_clean["ItemCode"].unique()), name="ItemCode")
all_dates = pd.date_range(train_clean["Date"].min(), train_clean["Date"].max(), freq="D", name="Date")

full_index = pd.MultiIndex.from_product([all_skus, all_dates], names=["ItemCode", "Date"])

daily_panel = (
    daily.set_index(["ItemCode", "Date"])
         .reindex(full_index)
         .reset_index()
)

fill_zero_cols = [
    "y_net", "y_gross", "y_return", "sales", "cost", "profit",
    "transaction_count", "y_net_clip"
]

for col in fill_zero_cols:
    daily_panel[col] = daily_panel[col].fillna(0)

print("daily_panel shape:", daily_panel.shape)
display(daily_panel.head())

daily_panel shape: (28014888, 10)


,ItemCode,Date,y_net,y_gross,y_return,sales,cost,profit,transaction_count,y_net_clip
0,SKU-00001,2020-11-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SKU-00001,2020-11-18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,SKU-00001,2020-11-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,SKU-00001,2020-11-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,SKU-00001,2020-11-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
n_skus = train_clean["ItemCode"].nunique()
n_days = (train_clean["Date"].max() - train_clean["Date"].min()).days + 1

print("n_skus:", n_skus)
print("n_days:", n_days)
print("expected rows:", n_skus * n_days)
print("actual rows:", len(daily_panel))

assert len(daily_panel) == n_skus * n_days

n_skus: 15972
n_days: 1754
expected rows: 28014888
actual rows: 28014888


### Add basic calender features

In [9]:
daily_panel["dayofweek"] = daily_panel["Date"].dt.dayofweek  # Monday=0, Sunday=6
daily_panel["is_saturday"] = (daily_panel["dayofweek"] == 5).astype(int)
daily_panel["is_sunday"] = (daily_panel["dayofweek"] == 6).astype(int)
daily_panel["month"] = daily_panel["Date"].dt.month
daily_panel["day"] = daily_panel["Date"].dt.day
daily_panel["weekofyear"] = daily_panel["Date"].dt.isocalendar().week.astype(int)
daily_panel["year"] = daily_panel["Date"].dt.year

display(daily_panel.head())

,ItemCode,Date,y_net,y_gross,y_return,sales,cost,profit,transaction_count,y_net_clip,dayofweek,is_saturday,is_sunday,month,day,weekofyear,year
0,SKU-00001,2020-11-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,11,17,47,2020
1,SKU-00001,2020-11-18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2,0,0,11,18,47,2020
2,SKU-00001,2020-11-19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3,0,0,11,19,47,2020
3,SKU-00001,2020-11-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,0,11,20,47,2020
4,SKU-00001,2020-11-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,0,11,21,47,2020


### 2.4 Data diagnostics

In [10]:
dow_summary = (
    daily_panel.groupby("dayofweek")
    .agg(
        total_y_net=("y_net", "sum"),
        avg_y_net_per_day=("y_net", "mean"),
        total_y_gross=("y_gross", "sum"),
        avg_y_gross_per_day=("y_gross", "mean"),
        active_rows=("y_gross", lambda x: (x > 0).sum())
    )
    .reset_index()
)

dow_map = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}

dow_summary["day_name"] = dow_summary["dayofweek"].map(dow_map)

display(dow_summary[["dayofweek", "day_name", "total_y_net", "avg_y_net_per_day", "total_y_gross", "avg_y_gross_per_day", "active_rows"]])

,dayofweek,day_name,total_y_net,avg_y_net_per_day,total_y_gross,avg_y_gross_per_day,active_rows
0,0,Monday,441516.0,0.110573,454178.0,0.113744,89418
1,1,Tuesday,408498.0,0.101896,423731.0,0.105696,84297
2,2,Wednesday,428302.0,0.106836,444114.0,0.110780,80993
3,3,Thursday,432126.0,0.107790,446238.0,0.111310,79471
4,4,Friday,427358.0,0.106600,443001.0,0.110502,79654
5,5,Saturday,308884.0,0.077356,322259.0,0.080706,71604
6,6,Sunday,569.0,0.000142,570.0,0.000143,17


Check active days by SKU

In [11]:
max_train_date = daily_panel["Date"].max()

# summary stats per SKU
sku_activity = (
    daily_panel.groupby("ItemCode", as_index=False)
    .agg(
        active_days=("y_gross", lambda x: int((x > 0).sum())),
        active_net_days=("y_net_clip", lambda x: int((x > 0).sum())),
        total_y_net=("y_net", "sum"),
        total_y_gross=("y_gross", "sum"),
        total_return=("y_return", "sum"),
        total_sales=("sales", "sum"),
        total_cost=("cost", "sum"),
        total_profit=("profit", "sum"),
        total_transactions=("transaction_count", "sum")
    )
)

# start date and end date of sales/transactions
sale_dates = (
    daily_panel[daily_panel["y_gross"] > 0]
    .groupby("ItemCode", as_index=False)
    .agg(
        first_sale_date=("Date", "min"),
        last_sale_date=("Date", "max")
    )
)

# 3. Start date and end date of transactions: calculate for both sales and returns
transaction_dates = (
    daily_panel[daily_panel["transaction_count"] > 0]
    .groupby("ItemCode", as_index=False)
    .agg(
        first_transaction_date=("Date", "min"),
        last_transaction_date=("Date", "max")
    )
)

# 4. merge with sku_activity
sku_activity = (
    sku_activity
    .merge(sale_dates, on="ItemCode", how="left")
    .merge(transaction_dates, on="ItemCode", how="left")
)

# 5. Recency features
sku_activity["has_ever_sold"] = sku_activity["last_sale_date"].notna().astype(int)

sku_activity["days_since_last_sale"] = (
    max_train_date - sku_activity["last_sale_date"]
).dt.days

# if sku havent have y_gross >0, make recency very large to indicate inactive
sku_activity["days_since_last_sale"] = sku_activity["days_since_last_sale"].fillna(9999).astype(int)

sku_activity["days_since_last_transaction"] = (
    max_train_date - sku_activity["last_transaction_date"]
).dt.days

sku_activity["days_since_last_transaction"] = sku_activity["days_since_last_transaction"].fillna(9999).astype(int)

# 6. Profit fields and ranking
sku_activity["positive_profit"] = sku_activity["total_profit"].clip(lower=0)

sku_activity["profit_rank"] = (
    sku_activity["positive_profit"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# 7. Return-rate feature
sku_activity["return_rate_qty"] = np.where(
    sku_activity["total_y_gross"] > 0,
    sku_activity["total_return"] / sku_activity["total_y_gross"],
    0
)

# 8. QA output
print("sku_activity shape:", sku_activity.shape)

display(sku_activity.head())

print("\nActive days describe:")
display(sku_activity["active_days"].describe())

print("\nDays since last sale describe:")
display(sku_activity["days_since_last_sale"].describe())

print("\nNumber of SKUs that never had positive sales:")
print((sku_activity["has_ever_sold"] == 0).sum())

print("\nTop 20 SKUs by positive profit:")
display(
    sku_activity
    .sort_values("positive_profit", ascending=False)
    .head(20)
)

sku_activity shape: (15972, 20)


,ItemCode,active_days,active_net_days,total_y_net,total_y_gross,total_return,total_sales,total_cost,total_profit,total_transactions,first_sale_date,last_sale_date,first_transaction_date,last_transaction_date,has_ever_sold,days_since_last_sale,days_since_last_transaction,positive_profit,profit_rank,return_rate_qty
0,SKU-00001,15,15,30.0,30.0,0.0,3.608433e+07,0.0,3.608433e+07,30.0,2025-05-26,2025-08-28,2025-05-26,2025-08-28,1,8,8,3.608433e+07,781,0.0
1,SKU-00002,895,895,5894.0,5894.0,0.0,8.012686e+09,0.0,8.012686e+09,5896.0,2022-01-10,2025-09-05,2022-01-10,2025-09-05,1,0,0,8.012686e+09,2,0.0
2,SKU-00003,1061,1061,10935.0,10935.0,0.0,1.670068e+10,0.0,1.670068e+10,10936.0,2022-01-03,2025-09-04,2022-01-03,2025-09-04,1,1,1,1.670068e+10,1,0.0
3,SKU-00004,279,279,659.0,659.0,0.0,8.359968e+08,0.0,8.359968e+08,659.0,2023-07-19,2024-12-20,2023-07-19,2024-12-20,1,259,259,8.359968e+08,12,0.0
4,SKU-00005,327,327,1101.0,1101.0,0.0,2.243340e+09,0.0,2.243340e+09,1103.0,2022-01-03,2023-06-26,2022-01-03,2023-06-26,1,802,802,2.243340e+09,4,0.0



Active days describe:


count    15972.000000
mean        30.394065
std         76.503322
min          0.000000
25%          2.000000
50%          6.000000
75%         22.000000
max       1061.000000
Name: active_days, dtype: float64


Days since last sale describe:


count    15972.000000
mean       379.001064
std        684.980664
min          0.000000
25%         52.000000
50%        205.000000
75%        546.250000
max       9999.000000
Name: days_since_last_sale, dtype: float64


Number of SKUs that never had positive sales:
58

Top 20 SKUs by positive profit:


,ItemCode,active_days,active_net_days,total_y_net,total_y_gross,total_return,total_sales,total_cost,total_profit,total_transactions,first_sale_date,last_sale_date,first_transaction_date,last_transaction_date,has_ever_sold,days_since_last_sale,days_since_last_transaction,positive_profit,profit_rank,return_rate_qty
2,SKU-00003,1061,1061,10935.0,10935.0,0.0,1.670068e+10,0.000000e+00,1.670068e+10,10936.0,2022-01-03,2025-09-04,2022-01-03,2025-09-04,1,1,1,1.670068e+10,1,0.000000
1,SKU-00002,895,895,5894.0,5894.0,0.0,8.012686e+09,0.000000e+00,8.012686e+09,5896.0,2022-01-10,2025-09-05,2022-01-10,2025-09-05,1,0,0,8.012686e+09,2,0.000000
9197,SKU-09458,82,82,62427.0,62430.0,3.0,1.008135e+10,7.440227e+09,2.641121e+09,154.0,2020-11-17,2024-04-24,2020-11-17,2024-04-24,1,499,499,2.641121e+09,3,0.000048
4,SKU-00005,327,327,1101.0,1101.0,0.0,2.243340e+09,0.000000e+00,2.243340e+09,1103.0,2022-01-03,2023-06-26,2022-01-03,2023-06-26,1,802,802,2.243340e+09,4,0.000000
8354,SKU-08589,53,53,29011.0,29011.0,0.0,4.882728e+09,3.597022e+09,1.285705e+09,106.0,2021-05-11,2025-05-15,2021-05-11,2025-05-15,1,113,113,1.285705e+09,5,0.000000
12229,SKU-12534,657,654,28349.0,28486.0,137.0,5.356791e+09,4.140987e+09,1.215804e+09,1556.0,2021-03-17,2025-09-04,2021-03-17,2025-09-04,1,1,1,1.215804e+09,6,0.004809
9492,SKU-09760,994,988,220421.0,222295.0,1874.0,5.971434e+09,4.776121e+09,1.195313e+09,3843.0,2022-01-03,2025-09-05,2022-01-03,2025-09-05,1,0,0,1.195313e+09,7,0.008430
12230,SKU-12537,676,674,24036.0,24127.0,91.0,4.576081e+09,3.469281e+09,1.106800e+09,1624.0,2021-03-17,2025-08-12,2021-03-17,2025-08-12,1,24,24,1.106800e+09,8,0.003772
315,SKU-00324,731,724,4710.0,4800.0,90.0,5.669205e+09,4.690830e+09,9.783747e+08,2488.0,2022-01-03,2025-02-13,2022-01-03,2025-02-13,1,204,204,9.783747e+08,9,0.018750
13993,SKU-14323,846,837,17918.0,18183.0,265.0,4.229909e+09,3.292805e+09,9.371040e+08,3536.0,2022-01-03,2025-09-05,2022-01-03,2025-09-05,1,0,0,9.371040e+08,10,0.014574


### 2.5 Save processed data

In [12]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

daily_panel_path = DATA_PROCESSED / "daily_panel.parquet"
sku_activity_path = DATA_PROCESSED / "sku_activity.parquet"

daily_panel.to_parquet(daily_panel_path, index=False)
sku_activity.to_parquet(sku_activity_path, index=False)

print("Saved:", daily_panel_path)
print("Saved:", sku_activity_path)

Saved: ../data/processed/daily_panel.parquet
Saved: ../data/processed/sku_activity.parquet
